# 01 · Extraction: from file bytes to blocks

Stage 1 of the ingestion pipeline (`app/ingestion/validate.py`, `app/ingestion/extract.py`),
run on the real FBR corpus. Each section shows one rule the code applies and the evidence that
it works. Nothing here calls the LLM except the single caption cell at the end.

In [1]:
# Run from the project root so app/ imports and corpus/ paths work.
import os, sys, time, warnings
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# Silence per-item progress bars from the model libraries so outputs stay readable.
os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 90)
print("project root:", Path.cwd().name)

project root: fbr-di-assistant


## 1. Validation: the real type comes from the bytes, not the name

`validate_file` checks magic bytes, encryption and zip bombs before anything is stored.

In [2]:
from app.exceptions import DomainError
from app.ingestion.validate import validate_file
from tests.file_factories import make_docx, make_pdf, make_pptx

samples = {
    "sro_69 (real PDF)": (Path("corpus/legal/sro_69(I)_2025_digital_invoicing_rules.pdf").read_bytes(), "sro.pdf"),
    "notes.docx": (make_docx(), "notes.docx"),
    "deck.pptx": (make_pptx(), "deck.pptx"),
    "exe renamed .pdf": (b"MZ\x90\x00" + b"\x00" * 64, "invoice.pdf"),
    "locked.pdf": (make_pdf(password="secret"), "locked.pdf"),
    "empty.txt": (b"", "empty.txt"),
}
rows = []
for label, (data, name) in samples.items():
    try:
        f = validate_file(data, name)
        rows.append({"file": label, "result": "accepted", "type": f.file_type, "pages": f.page_count, "sha256": f.sha256[:12]})
    except DomainError as err:
        rows.append({"file": label, "result": f"{err.status_code} {err.code}", "type": "", "pages": None, "sha256": err.message})
pd.DataFrame(rows)

,file,result,type,pages,sha256
0,sro_69 (real PDF),accepted,pdf,11.0,ad86d25b893e
1,notes.docx,accepted,docx,NaN,099066398f6f
2,deck.pptx,accepted,pptx,2.0,5136bc0904a3
3,exe renamed .pdf,415 unsupported_file_type,,NaN,The file content does not match its .pdf extension.
4,locked.pdf,400 invalid_document,,NaN,The PDF is password-protected; upload an unlocked copy.
5,empty.txt,400 invalid_document,,NaN,The file is empty.


## 2. Which reader handles which page

Measured on this corpus: PyMuPDF reads a text-layer page in milliseconds; Docling needed
8-12 s per page on CPU. So PyMuPDF reads every page with text, and pages without text go to
a caption (mostly images) or OCR (otherwise).

In [3]:
from app.ingestion.caption import FakeCaptioner
from app.ingestion.extract import Extractor

extractor = Extractor(FakeCaptioner())  # fake captioner: no API calls in this overview
files = ["legal/sales_tax_rules_2006_upto_2026-07-31.pdf", "legal/sro_69(I)_2025_digital_invoicing_rules.pdf",
         "technical/di_api_technical_spec_v1.12.pdf", "technical/di_user_manual_v1.5.pdf", "portal/iris_dashboard_manual.pdf"]
results, rows = {}, []
for rel in files:
    path = Path("corpus") / rel
    t0 = time.time()
    result = await extractor.extract(validate_file(path.read_bytes(), path.name))
    took = time.time() - t0
    results[path.name] = result
    kinds = pd.Series([b.kind for b in result.blocks]).value_counts().to_dict()
    rows.append({"document": path.name, "pages": result.page_count, "blocks": len(result.blocks),
                 "kinds": kinds, "ocr/caption pages": result.ocr_pages,
                 "blocks marked ocr": sum(b.ocr for b in result.blocks), "seconds": round(took, 1)})
pd.DataFrame(rows)

,document,pages,blocks,kinds,ocr/caption pages,blocks marked ocr,seconds
0,sales_tax_rules_2006_upto_2026-07-31.pdf,232,3039,"{'text': 2980, 'table': 59}",0,0,46.1
1,sro_69(I)_2025_digital_invoicing_rules.pdf,11,407,{'text': 407},0,407,0.2
2,di_api_technical_spec_v1.12.pdf,51,321,"{'text': 270, 'table': 51}",0,0,31.4
3,di_user_manual_v1.5.pdf,21,224,"{'text': 222, 'table': 2}",0,0,9.4
4,iris_dashboard_manual.pdf,14,14,{'caption': 14},14,14,1.6


**Read the table:** the 232-page Rules take about a minute (mostly table detection), the
14-page IRIS manual is 14 caption pages (pure screenshots), and SRO 69 is a scan whose blocks
are all marked `ocr` because its text layer came from someone else's OCR.

## 3. Running headers, footers and footnote markers are removed

Rules page 108 (printed "(91)"): the raw text layer starts with the running header and has the
printed page number at the bottom. The extracted blocks keep only the content.

In [4]:
import pymupdf

rules = results["sales_tax_rules_2006_upto_2026-07-31.pdf"]
with pymupdf.open("corpus/legal/sales_tax_rules_2006_upto_2026-07-31.pdf") as doc:
    raw = doc[107].get_text()
print("RAW first line          :", raw.strip().splitlines()[0])
print("'(91)' in the raw page  :", "(91)" in raw)
page_108 = [b for b in rules.blocks if b.page == 108]
text_108 = "\n".join(b.text for b in page_108)
print("\n'Sales Tax Rules, 2006' still in blocks:", "Sales Tax Rules, 2006" in text_108)
print("'(91)' still in blocks:", "(91)" in text_108)
print("section path:", page_108[0].section_path)

RAW first line          : Sales Tax Rules, 2006 
'(91)' in the raw page  : True

'Sales Tax Rules, 2006' still in blocks: False
'(91)' still in blocks: False
section path: ('CHAPTER XIV PROCEDURE FOR LICENSING, ISSUANCE OF ELECTRONIC SALES TAX INVOICES AND INTEGRATION OF REGISTERED PERSONS',)


## 4. Amendment footnotes become metadata

Rule 150Q(3) carries superscript marker 252; its footnote says
*"Sub-rule (3) of Rule 150Q added vide S.R.O. 2071(I)/2025"*. The marker is removed from the
text (it would put a stray "252" next to section numbers) and the SRO lands in `amended_by`.

In [5]:
block = next(b for b in rules.blocks if b.page == 105 and "Retailers" in b.text)
print("text starts :", block.text[:110].replace("\n", " "))
print("'252' in text:", "252" in block.text)
print("amended_by  :", block.amended_by)
amended = [b for b in rules.blocks if b.amended_by]
print(f"\n{len(amended)} of {len(rules.blocks)} Rules blocks carry amendment metadata, e.g.:")
pd.Series([ref for b in amended for ref in b.amended_by]).value_counts().head(8)

text starts : [(3) Retailers whose deductible withholding tax under sections 236G or 236H of the Income
'252' in text: False
amended_by  : ('S.R.O. 2071(I)/2025',)

254 of 3039 Rules blocks carry amendment metadata, e.g.:


S.R.O. 918(I)/2019     51
S.R.O. 350(I)/2024     20
S.R.O. 494(I)/2015     17
S.R.O. 610(I)/2010     15
S.R.O. 821(I)/2010     15
S.R.O. 530(I)/2008     14
S.R.O. 1507(I)/2024    13
S.R.O. 1203(I)/2019     8
Name: count, dtype: int64

## 5. Tables: cells from the page text, header carried over pages

The error-code table starts on page 36 and continues to page 45 without repeating its header.
Each continuation page gets the last real header, so every row still says what its columns mean.

In [6]:
spec = results["di_api_technical_spec_v1.12.pdf"]
tables = [b for b in spec.blocks if b.kind == "table"]
print(f"{len(tables)} tables in the API spec\n")
page_38 = next(b for b in tables if b.page == 38)
print("page 38 (a continuation page), first rows:")
print("\n".join(page_38.text.splitlines()[:5]))

51 tables in the API spec

page 38 (a continuation page), first rows:
|MESSAGE CODE|MESSAGE_DESC|BRIEF_MSG_DESC|
|---|---|---|
|0042|Provide invoice date.|Invoice date cannot be empty, please provide invoice date.|
|0043|Provide valid Date.|Invoice date is not valid, please provide valid invoice date.|
|0044|Provide HS Code.|HS Code cannot be empty, please provide HS Code|


## 6. Headings become the section path

Headings are found by recurring font size (Rules: 14 pt chapters) or numbered style
(spec: "5.1." in a different font). One-off cover titles and the table of contents are ignored.

In [7]:
paths = []
for b in spec.blocks:
    if b.section_path and b.section_path not in paths:
        paths.append(b.section_path)
for p in paths[8:20]:
    print(" > ".join(p))

4. Digital Invoicing API > 4.1. Web method for Digital Invoicing - Post > 4.1.2. Sample JSON Data (For Production):
4. Digital Invoicing API > 4.1. Web method for Digital Invoicing - Post > 4.1.3. Sample JSON Valid Response:
4. Digital Invoicing API > 4.1. Web method for Digital Invoicing - Post > 4.1.4. Sample JSON Invalid Response 1:
4. Digital Invoicing API > 4.1. Web method for Digital Invoicing - Post > 4.1.5. Sample JSON Invalid Response 2:
4. Digital Invoicing API > 4.1. Web method for Digital Invoicing - Post > 4.1.6. HTTP Status Codes:
4. Digital Invoicing API > 4.2. Web method for Digital Invoicing - Validate
4. Digital Invoicing API > 4.2. Web method for Digital Invoicing - Validate > 4.2.1. Sample JSON Data (For Sandbox Only):
4. Digital Invoicing API > 4.2. Web method for Digital Invoicing - Validate > 4.2.2. Sample JSON Data (For Production):
4. Digital Invoicing API > 4.2. Web method for Digital Invoicing - Validate > 4.2.3. Sample Json Response (In Case of Valid):
4. Di

## 7. Image-only pages are captioned by the small Claude model

The IRIS manual has no text layer at all. One page is captioned for real here (about $0.001
with Haiku). If `ANTHROPIC_API_KEY` in `.env` is not a valid key, the cell says so and skips.

In [8]:
from app.exceptions import UpstreamUnavailableError
from app.ingestion.caption import ClaudeCaptioner
from app.ingestion.extract import render_pages_png

iris = Path("corpus/portal/iris_dashboard_manual.pdf").read_bytes()
png = render_pages_png(iris, [3])[3]
print(f"page 3 rendered: {len(png) / 1024:.0f} KB PNG")
try:
    t0 = time.time()
    caption = await ClaudeCaptioner().caption(png, 3, "iris_dashboard_manual.pdf")
    print(f"caption in {time.time() - t0:.1f}s:\n\n{caption}")
except UpstreamUnavailableError:
    print("SKIPPED: the Anthropic API rejected the request (check ANTHROPIC_API_KEY in .env); "
          "rerun this cell once a valid key is set.")

page 3 rendered: 432 KB PNG


Anthropic rejected the API key; check ANTHROPIC_API_KEY in .env


SKIPPED: the Anthropic API rejected the request (check ANTHROPIC_API_KEY in .env); rerun this cell once a valid key is set.
